# Temporal Price vs. Time Series Diagrams for All Models
### CSE 4112: Machine Learning Laboratory Final Project
**BD-FPP: Bangladesh Food-Price Prediction**

This notebook generates publication-quality **Price vs. Time** series diagrams for all machine learning regression models trained on the **chronological (temporal) split** using the raw Weka buffer outputs.

---

### Key Requirements Addressed:
1. **Actual vs. Predicted Price:** Both trajectories plotted across calendar dates.
2. **Train & Test Period Demarcation:** Shaded background regions for **Train Period (80%)** and **Test Period (20%)**, with an explicit split boundary line (`2026-05-04`).
3. **Multi-Model Coverage:** Gradient Boosting (`gbr`), Linear Regression (`lr`), M5P Model Tree (`m5p`), Random Forest (`rf`), and XGBoost (`xgbr`).
4. **Multi-Horizon Coverage:** 7-day, 14-day, and 30-day forecast targets.
5. **Commodity Strategy ("Should I do this for all commodities?"):**
   - **Recommendation:** In academic papers and presentations, plotting all 18 commodities creates an unreadable volume of 90–270 graphs. The standard methodology is to highlight **5–6 key representative staple commodities** covering major food groups (Staples, Protein, Cooking Oils, Volatile Vegetables) in the main text.
   - For complete completeness, this notebook also includes an **Automated Batch Export** cell that generates and saves plots for **all 18 commodities** into categorized folders (`timeseries_figures/`).


In [ ]:
import os
import io
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import pearsonr

# Set high-resolution plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 150

# Ensure working directory is set to project root or evaluation folder
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..')) if os.path.exists('../../Buffer') else os.getcwd()
BUFFER_DIR = os.path.join(BASE_DIR, 'Buffer', 'temporal')
PREPROC_DIR = os.path.join(BASE_DIR, 'Preprocessing')
OUTPUT_DIR = os.path.join(BASE_DIR, 'Evaluation', 'temporal', 'timeseries_figures')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Base Directory: {BASE_DIR}")
print(f"Buffer Directory: {BUFFER_DIR}")
print(f"Output Directory for Figures: {OUTPUT_DIR}")


## 1. Weka Buffer Parsing & Dataset Alignment

Weka logs prediction tables under `=== Predictions on {training/test} set ===`.
Each row maps 1-to-1 with the chronological train (`moa_train_80_lag.csv`, 20,165 rows) and test (`moa_test_20_lag.csv`, 5,042 rows) datasets.


In [ ]:
def parse_buffer_file(path):
    """Extracts the CSV table of predictions from raw Weka run logs."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Buffer file not found: {path}")
        
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        lines = f.readlines()
        
    start_idx = None
    for i, line in enumerate(lines):
        if line.strip().startswith('inst#,actual,predicted,error'):
            start_idx = i
            break
            
    if start_idx is None:
        raise ValueError(f"Could not find predictions header in {path}")
    
    data_lines = [lines[start_idx]]
    for line in lines[start_idx + 1:]:
        stripped = line.strip()
        if not stripped or stripped.startswith('==='):
            break
        data_lines.append(line)
        
    df = pd.read_csv(io.StringIO(''.join(data_lines)))
    df.columns = df.columns.str.strip()
    df['commodity_name'] = df['commodity_name'].astype(str).str.strip().str.strip("'")
    return df

# Model definition mappings
MODEL_MAP = {
    'rf': {'name': 'Random Forest', 'color': '#2ca02c'},
    'xgbr': {'name': 'XGBoost', 'color': '#9467bd'},
    'gbr': {'name': 'Gradient Boosting', 'color': '#d62728'},
    'lr': {'name': 'Linear Regression', 'color': '#1f77b4'},
    'm5p': {'name': 'M5P Model Tree', 'color': '#ff7f0e'}
}

# Load reference datasets once
train_ref = pd.read_csv(os.path.join(PREPROC_DIR, 'moa_train_80_lag.csv'))
test_ref = pd.read_csv(os.path.join(PREPROC_DIR, 'moa_test_20_lag.csv'))

train_ref['date'] = pd.to_datetime(train_ref[['year', 'month', 'day']])
test_ref['date'] = pd.to_datetime(test_ref[['year', 'month', 'day']])

SPLIT_DATE = test_ref['date'].min()
print(f"Loaded Train: {len(train_ref)} rows ({train_ref['date'].min().date()} to {train_ref['date'].max().date()})")
print(f"Loaded Test:  {len(test_ref)} rows ({test_ref['date'].min().date()} to {test_ref['date'].max().date()})")
print(f"Split Boundary Date: {SPLIT_DATE.date()}")


## 2. Assembling Full Time-Series for Any Model & Horizon

Combines train and test predictions with dates, commodities, retail units, and divisions.


In [ ]:
def get_model_timeseries_df(model_key, horizon=7):
    """Loads train and test buffer predictions for a given model and horizon,
    aligning them with calendar dates and commodities."""
    train_folder = f"target_{horizon}" if horizon != 30 else "target-30"
    test_folder = f"target_{horizon}"
    
    train_path = os.path.join(BUFFER_DIR, 'train', train_folder, f"{model_key}_buffer_train_{horizon}")
    test_path = os.path.join(BUFFER_DIR, 'test', test_folder, f"{model_key}_buffer_{horizon}")
    
    train_buf = parse_buffer_file(train_path)
    test_buf = parse_buffer_file(test_path)
    
    target_col = f"target_{horizon}d"
    
    df_train = train_ref.copy()
    df_train['predicted'] = train_buf['predicted']
    df_train['actual_target'] = df_train[target_col]
    df_train['split'] = 'Train'
    
    df_test = test_ref.copy()
    df_test['predicted'] = test_buf['predicted']
    df_test['actual_target'] = df_test[target_col]
    df_test['split'] = 'Test'
    
    full_df = pd.concat([df_train, df_test], ignore_index=True)
    return full_df

print("Pipeline verified. Available models:", list(MODEL_MAP.keys()))


## 3. Publication-Quality Plotting Function

Renders Price vs. Time with:
- Solid blue curve: Ground truth actual prices
- Dashed colored curve: Model predicted prices
- Shaded green region: Training phase (first 80% chronologically)
- Shaded orange region: Out-of-sample Testing phase (last 20%)
- Vertical boundary line with date label
- Dynamic retail unit display (BDT / Kilogram, BDT / 4 Pcs, etc.)
- Test partition metric box ($R$, RMSE, MAPE %)


In [ ]:
def plot_commodity_timeseries(
    model_key='rf',
    horizon=7,
    commodity='Rice - Medium',
    division=None,
    save=True,
    show=True
):
    """Generates a Price vs. Time diagram for a specified model, horizon, and commodity."""
    model_info = MODEL_MAP[model_key]
    model_name = model_info['name']
    model_color = model_info['color']
    
    full_df = get_model_timeseries_df(model_key, horizon)
    
    # Filter by commodity
    sub = full_df[full_df['commodity_name'] == commodity].copy()
    if sub.empty:
        print(f"Error: Commodity '{commodity}' not found.")
        return None
        
    unit = sub['retail_unit'].iloc[0]
    
    # Optional filter by division or calculate national daily mean
    if division is not None:
        sub = sub[sub['division'] == division]
        title_region = f"[{division} Division]"
    else:
        title_region = "[National Average across Divisions]"
        
    daily = sub.groupby(['date', 'split'], as_index=False).agg({
        'actual_target': 'mean',
        'predicted': 'mean'
    }).sort_values('date')
    
    # Compute test set performance metrics
    test_data = daily[daily['split'] == 'Test']
    if len(test_data) > 1:
        act_te = test_data['actual_target'].values
        pred_te = test_data['predicted'].values
        rmse = np.sqrt(np.mean((act_te - pred_te) ** 2))
        r_val, _ = pearsonr(act_te, pred_te) if np.std(act_te) > 0 and np.std(pred_te) > 0 else (np.nan, 0)
        mape = np.mean(np.abs((act_te - pred_te) / act_te)) * 100
        metric_str = f"Test Metrics:\nPearson $R$: {r_val:.4f}\nRMSE: {rmse:.2f}\nMAPE: {mape:.2f}%"
    else:
        metric_str = "Test Metrics: N/A"
        
    # Plotting
    fig, ax = plt.subplots(figsize=(13, 5.5), dpi=150)
    
    # Plot Actual & Predicted curves
    ax.plot(daily['date'], daily['actual_target'], label='Actual Price (Ground Truth)', 
            color='#1f77b4', linewidth=2.0, alpha=0.9)
    ax.plot(daily['date'], daily['predicted'], label=f'Predicted Price ({model_name})', 
            color=model_color, linewidth=1.8, linestyle='--', alpha=0.95)
    
    # Train / Test split shading
    min_date = daily['date'].min()
    max_date = daily['date'].max()
    ax.axvspan(min_date, SPLIT_DATE, color='#2ca02c', alpha=0.10, label='Train Period (Earliest 80% Chronological)')
    ax.axvspan(SPLIT_DATE, max_date, color='#ff7f0e', alpha=0.12, label='Test Period (Recent 20% Out-of-Sample)')
    ax.axvline(SPLIT_DATE, color='#333333', linestyle=':', linewidth=1.6, 
               label=f'Temporal Split Boundary ({SPLIT_DATE.strftime("%Y-%m-%d")})')
    
    # Metric textbox
    props = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='#bbbbbb')
    ax.text(0.985, 0.05, metric_str, transform=ax.transAxes, fontsize=9.5,
            verticalalignment='bottom', horizontalalignment='right', bbox=props)
    
    # Formatting
    ax.set_title(f'Time-Series Forecasting: Price vs. Time\n{commodity} ({unit}) -- {model_name} ({horizon}-Day Forecast Horizon) {title_region}',
                 fontsize=12, fontweight='bold', pad=12)
    ax.set_xlabel('Time (Calendar Date)', fontsize=10.5, labelpad=8)
    ax.set_ylabel(f'Price (BDT / {unit})', fontsize=10.5, labelpad=8)
    
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    fig.autofmt_xdate()
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper left', framealpha=0.95, fontsize=9)
    fig.tight_layout()
    
    if save:
        clean_name = re.sub(r'[^a-zA-Z0-9_-]', '_', commodity)
        save_folder = os.path.join(OUTPUT_DIR, model_key, f"target_{horizon}")
        os.makedirs(save_folder, exist_ok=True)
        filename = f"{model_key}_{clean_name}_{horizon}d.png"
        out_file = os.path.join(save_folder, filename)
        fig.savefig(out_file, bbox_inches='tight')
        print(f"Saved: {out_file}")
        
    if show:
        plt.show()
    else:
        plt.close(fig)
        
    return fig


## 4. Key Representative Staples Showcase (Recommended Presentation)

To address the question: **"Should I do this for all the commodities?"**
* **In your report / presentation:** Presenting the top 5–6 staple items representing diverse agricultural market sectors is best practice:
  1. **Cereals / Carbohydrates:** `Rice - Medium`, `Ata (Packet)`
  2. **Poultry & Livestock:** `Broiler chicken`, `Egg Farm-Red`
  3. **Edible Oils:** `Soybean Oil(loose)`
  4. **Volatile Produce & Spices:** `Onion (local)`


In [ ]:
REPRESENTATIVE_COMMODITIES = [
    'Rice - Medium',
    'Ata (Packet)',
    'Broiler chicken',
    'Egg Farm-Red',
    'Soybean Oil(loose)',
    'Onion (local)'
]

print("Generating representative staple time-series diagrams for the winning model (Random Forest, 7-Day Horizon)...")
for com in REPRESENTATIVE_COMMODITIES:
    plot_commodity_timeseries(model_key='rf', horizon=7, commodity=com, save=True, show=True)


## 5. Multi-Model Comparison on the Same Commodity

Compare how all five models (**Random Forest**, **XGBoost**, **Gradient Boosting**, **Linear Regression**, and **M5P**) perform on the same commodity time-series.


In [ ]:
def plot_all_models_on_commodity(commodity='Rice - Medium', horizon=7, save=True):
    """Generates a 5-panel subplot comparing all 5 algorithms on one commodity."""
    fig, axes = plt.subplots(5, 1, figsize=(14, 18), sharex=True, dpi=140)
    
    for idx, (m_key, m_info) in enumerate(MODEL_MAP.items()):
        ax = axes[idx]
        full_df = get_model_timeseries_df(m_key, horizon)
        sub = full_df[full_df['commodity_name'] == commodity]
        unit = sub['retail_unit'].iloc[0]
        
        daily = sub.groupby(['date', 'split'], as_index=False).agg({
            'actual_target': 'mean',
            'predicted': 'mean'
        }).sort_values('date')
        
        # Test metrics
        test_data = daily[daily['split'] == 'Test']
        r_val, _ = pearsonr(test_data['actual_target'], test_data['predicted'])
        rmse = np.sqrt(np.mean((test_data['actual_target'] - test_data['predicted']) ** 2))
        
        # Curves
        ax.plot(daily['date'], daily['actual_target'], label='Actual Price', color='#1f77b4', linewidth=1.8)
        ax.plot(daily['date'], daily['predicted'], label=f'{m_info["name"]}', color=m_info['color'], 
                linewidth=1.6, linestyle='--')
        
        # Shading
        ax.axvspan(daily['date'].min(), SPLIT_DATE, color='#2ca02c', alpha=0.08)
        ax.axvspan(SPLIT_DATE, daily['date'].max(), color='#ff7f0e', alpha=0.10)
        ax.axvline(SPLIT_DATE, color='#333333', linestyle=':', linewidth=1.2)
        
        ax.set_ylabel(f'BDT / {unit}', fontsize=9.5)
        ax.set_title(f'{m_info["name"]} (Test $R = {r_val:.3f}$, RMSE = {rmse:.2f})', 
                     fontsize=10.5, fontweight='bold', loc='left')
        ax.legend(loc='upper left', fontsize=8.5)
        ax.grid(True, linestyle='--', alpha=0.5)
        
    axes[-1].set_xlabel('Time (Calendar Date)', fontsize=10.5, labelpad=8)
    axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    fig.autofmt_xdate()
    
    fig.suptitle(f'Cross-Model Trajectory Comparison: {commodity} ({horizon}-Day Target Horizon)',
                 fontsize=14, fontweight='bold', y=0.995)
    fig.tight_layout()
    
    if save:
        clean_name = re.sub(r'[^a-zA-Z0-9_-]', '_', commodity)
        out_file = os.path.join(OUTPUT_DIR, f"cross_model_comparison_{clean_name}_{horizon}d.png")
        fig.savefig(out_file, bbox_inches='tight')
        print(f"Saved cross-model grid: {out_file}")
        
    plt.show()

plot_all_models_on_commodity('Rice - Medium', horizon=7)


## 6. Multi-Horizon Forecast Comparison (7-Day vs. 14-Day vs. 30-Day)

Inspect forecast degradation as the forecasting horizon expands.


In [ ]:
def plot_multi_horizon_comparison(model_key='rf', commodity='Soybean Oil(loose)', save=True):
    """Plots 7-day, 14-day, and 30-day forecasts side by side for a model."""
    fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True, dpi=140)
    horizons = [7, 14, 30]
    m_info = MODEL_MAP[model_key]
    
    for idx, h in enumerate(horizons):
        ax = axes[idx]
        full_df = get_model_timeseries_df(model_key, h)
        sub = full_df[full_df['commodity_name'] == commodity]
        unit = sub['retail_unit'].iloc[0]
        
        daily = sub.groupby(['date', 'split'], as_index=False).agg({
            'actual_target': 'mean',
            'predicted': 'mean'
        }).sort_values('date')
        
        test_data = daily[daily['split'] == 'Test']
        r_val, _ = pearsonr(test_data['actual_target'], test_data['predicted'])
        rmse = np.sqrt(np.mean((test_data['actual_target'] - test_data['predicted']) ** 2))
        
        ax.plot(daily['date'], daily['actual_target'], label='Actual Price', color='#1f77b4', linewidth=1.8)
        ax.plot(daily['date'], daily['predicted'], label=f'Predicted ({h}-day horizon)', color=m_info['color'], 
                linewidth=1.6, linestyle='--')
        
        ax.axvspan(daily['date'].min(), SPLIT_DATE, color='#2ca02c', alpha=0.08)
        ax.axvspan(SPLIT_DATE, daily['date'].max(), color='#ff7f0e', alpha=0.10)
        ax.axvline(SPLIT_DATE, color='#333333', linestyle=':', linewidth=1.2)
        
        ax.set_ylabel(f'BDT / {unit}', fontsize=9.5)
        ax.set_title(f'{h}-Day Forecast Horizon (Test $R = {r_val:.3f}$, RMSE = {rmse:.2f})', 
                     fontsize=10.5, fontweight='bold', loc='left')
        ax.legend(loc='upper left', fontsize=8.5)
        ax.grid(True, linestyle='--', alpha=0.5)
        
    axes[-1].set_xlabel('Time (Calendar Date)', fontsize=10.5, labelpad=8)
    axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    fig.autofmt_xdate()
    
    fig.suptitle(f'{m_info["name"]}: Multi-Horizon Forecasting Comparison on {commodity}',
                 fontsize=13, fontweight='bold', y=0.995)
    fig.tight_layout()
    
    if save:
        clean_name = re.sub(r'[^a-zA-Z0-9_-]', '_', commodity)
        out_file = os.path.join(OUTPUT_DIR, f"{model_key}_multi_horizon_{clean_name}.png")
        fig.savefig(out_file, bbox_inches='tight')
        print(f"Saved multi-horizon grid: {out_file}")
        
    plt.show()

plot_multi_horizon_comparison(model_key='rf', commodity='Soybean Oil(loose)')


## 7. Automated Batch Generator (All 18 Commodities × All Models)

Run this cell if you wish to generate and export **all 18 commodities** into organized disk directories (`timeseries_figures/<model_name>/target_<horizon>/`).


In [ ]:
def batch_generate_all(models_to_run=['rf', 'xgbr', 'gbr'], horizons_to_run=[7]):
    """Generates and saves Price vs. Time plots for all commodities."""
    all_commodities = sorted(train_ref['commodity_name'].unique().tolist())
    print(f"Starting batch generation for {len(all_commodities)} commodities across {len(models_to_run)} models...")
    
    count = 0
    for m in models_to_run:
        for h in horizons_to_run:
            print(f"\nProcessing Model: {MODEL_MAP[m]['name']} | Horizon: {h} days")
            for com in all_commodities:
                plot_commodity_timeseries(model_key=m, horizon=h, commodity=com, save=True, show=False)
                count += 1
                
    print(f"\nBatch generation complete! Successfully generated {count} diagrams in '{OUTPUT_DIR}'.")

# Example usage (Uncomment below to run the batch export for the top 3 models):
# batch_generate_all(models_to_run=['rf', 'xgbr', 'gbr'], horizons_to_run=[7])
print("Batch generator loaded. Call `batch_generate_all()` when you are ready to export all figures.")
